## Combine the Filter 3 JSON with existing

In [1]:
import json

In [2]:
with open('momentum_historical_stock_generation_14_10_2025_v1.json','r') as file:
    main_list = json.load(file)

In [3]:
main_list.keys()

dict_keys(['Aug 2014', 'Jul 2014', 'Sep 2014', 'Oct 2014', 'Nov 2014', 'Dec 2014', 'Jan 2015', 'Feb 2015', 'Mar 2015', 'Apr 2015', 'May 2015', 'Jun 2015', 'Jul 2015', 'Aug 2015', 'Sep 2015', 'Oct 2015', 'Nov 2015', 'Dec 2015', 'Jan 2016', 'Feb 2016', 'Mar 2016', 'Apr 2016', 'May 2016', 'Jun 2016', 'Jul 2016', 'Aug 2016', 'Sep 2016', 'Oct 2016', 'Nov 2016', 'Dec 2016', 'Jan 2017', 'Feb 2017', 'Mar 2017', 'Apr 2017', 'May 2017', 'Jun 2017', 'Jul 2017', 'Aug 2017', 'Sep 2017', 'Oct 2017', 'Nov 2017', 'Dec 2017', 'Jan 2018', 'Feb 2018', 'Mar 2018', 'Apr 2018', 'May 2018', 'Jun 2018', 'Jul 2018', 'Aug 2018', 'Sep 2018', 'Oct 2018', 'Nov 2018', 'Dec 2018', 'Jan 2019', 'Feb 2019', 'Mar 2019', 'Apr 2019', 'May 2019', 'Jun 2019', 'Jul 2019', 'Aug 2019', 'Sep 2019', 'Nov 2019', 'Dec 2019', 'Jan 2020', 'Oct 2019', 'Feb 2020', 'Mar 2020', 'Apr 2020', 'May 2020', 'Jun 2020', 'Jul 2020', 'Aug 2020', 'Sep 2020', 'Oct 2020', 'Nov 2020', 'Dec 2020', 'Jan 2021', 'Feb 2021', 'Mar 2021', 'Apr 2021', 'May 

In [4]:
with open('momentum_historical_stocks_generation_filter3_10_percent_16_10_2025_v2.json','r') as file:
    filter3_performance_stocks_list = json.load(file)

In [7]:
filter3_performance_stocks_list.keys() == main_list.keys()

True

In [8]:
for month_year, data in main_list.items():
    data['Filter 3'] = filter3_performance_stocks_list[month_year]['Filter 3']

In [10]:
main_list['Nov 2020']['Filter 3']

[['TATASTEEL.NS',
  'TRIDENT.NS',
  'POLYPLEX.NS',
  'AARTIIND.NS',
  'ALKYLAMINE.NS',
  'ESCORTS.NS'],
 ['POLYPLEX.NS',
  'AARTIIND.NS',
  'ALKYLAMINE.NS',
  'ESCORTS.NS',
  'TATASTEEL.NS',
  'TRIDENT.NS'],
 ['POLYPLEX.NS',
  'ALKYLAMINE.NS',
  'ESCORTS.NS',
  'TATASTEEL.NS',
  'TRIDENT.NS',
  'AARTIIND.NS'],
 ['POLYPLEX.NS',
  'ALKYLAMINE.NS',
  'ESCORTS.NS',
  'TATASTEEL.NS',
  'BHARATRAS.NS',
  'TRIDENT.NS'],
 ['POLYPLEX.NS',
  'ALKYLAMINE.NS',
  'ESCORTS.NS',
  'TATASTEEL.NS',
  'BHARATRAS.NS',
  'TRIDENT.NS'],
 ['POLYPLEX.NS',
  'ALKYLAMINE.NS',
  'ESCORTS.NS',
  'TATASTEEL.NS',
  'BHARATRAS.NS',
  'TRIDENT.NS'],
 ['POLYPLEX.NS',
  'ALKYLAMINE.NS',
  'ESCORTS.NS',
  'TATASTEEL.NS',
  'BHARATRAS.NS',
  'TRIDENT.NS'],
 ['POLYPLEX.NS',
  'ALKYLAMINE.NS',
  'ESCORTS.NS',
  'TATASTEEL.NS',
  'BHARATRAS.NS',
  'TRIDENT.NS'],
 ['POLYPLEX.NS',
  'ALKYLAMINE.NS',
  'ESCORTS.NS',
  'TATASTEEL.NS',
  'BHARATRAS.NS',
  'TRIDENT.NS'],
 ['POLYPLEX.NS',
  'ALKYLAMINE.NS',
  'ESCORTS.NS',
  'TAT

In [11]:
with open('temp.json','w') as file:
    json.dump(main_list,file,indent=4)

## Seperating into 10 files

In [18]:
import json
import copy
import os

# === Load the original JSON data ===
input_filename = 'temp.json'
if not os.path.exists(input_filename):
    print(f"Error: '{input_filename}' not found. Make sure the file is in the same directory as the script.")
    exit()

with open(input_filename, 'r') as file:
    original_data = json.load(file)

# === Create output folder if not exists ===
os.makedirs('./result_json', exist_ok=True)

# === Process 10 iterations ===
final = []
rows_for_dataframe = []

for i in range(10):
    output_data_for_file = {}

    for month_year, month_data in original_data.items():
        temp_month_data = copy.deepcopy(month_data)

        # --- Handle "Filter 3" robustly ---
        filter3_data = temp_month_data.get("Filter 3", [])

        # Case A: Filter 3 is a list of strings (single stock list)
        if all(isinstance(x, str) for x in filter3_data):
            temp_month_data["Filter 3"] = filter3_data  # same list for all iterations

        # Case B: Filter 3 is a list of lists (multiple iterations)
        elif all(isinstance(x, list) for x in filter3_data) and len(filter3_data) > 0:
            if i < len(filter3_data):
                temp_month_data["Filter 3"] = filter3_data[i]
            else:
                # If fewer than 10 lists, repeat the last one
                temp_month_data["Filter 3"] = filter3_data[-1]

        # Case C: Empty or malformed
        else:
            temp_month_data["Filter 3"] = []

        # --- Build row for DataFrame use ---
        row = {
            'iteration': i,
            'month_year': month_year
        }
        row.update(temp_month_data)
        rows_for_dataframe.append(row)

        output_data_for_file[month_year] = temp_month_data

    # === Write output file ===
    final.append(output_data_for_file)
    output_filename = f'./result_json/momentum_historical_stocks_generation_filter3_10_percent_16_10_2025_v2_{i}.json'
    with open(output_filename, 'w') as outfile:
        json.dump(output_data_for_file, outfile, indent=4)

    print(f"✅ Successfully created {output_filename}")

print("\n🎯 Process complete. 10 files have been generated.")


✅ Successfully created ./result_json/momentum_historical_stocks_generation_filter3_10_percent_16_10_2025_v2_0.json
✅ Successfully created ./result_json/momentum_historical_stocks_generation_filter3_10_percent_16_10_2025_v2_1.json
✅ Successfully created ./result_json/momentum_historical_stocks_generation_filter3_10_percent_16_10_2025_v2_2.json
✅ Successfully created ./result_json/momentum_historical_stocks_generation_filter3_10_percent_16_10_2025_v2_3.json
✅ Successfully created ./result_json/momentum_historical_stocks_generation_filter3_10_percent_16_10_2025_v2_4.json
✅ Successfully created ./result_json/momentum_historical_stocks_generation_filter3_10_percent_16_10_2025_v2_5.json
✅ Successfully created ./result_json/momentum_historical_stocks_generation_filter3_10_percent_16_10_2025_v2_6.json
✅ Successfully created ./result_json/momentum_historical_stocks_generation_filter3_10_percent_16_10_2025_v2_7.json
✅ Successfully created ./result_json/momentum_historical_stocks_generation_filte

## Visaulisation

In [ ]:
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# === Load Experimental Run Data ===
runs_file = "./om-results/Test4.1/value_strategy_performance_exp_3.1_Filter 3_all_runs.json"
with open(runs_file, "r") as f:
    runs_data = json.load(f)

runs = list(runs_data.keys())

# === Load Baseline Data ===
baseline_perf_file = "./value_strategy_performance_exp_03_Filter 3.json"
baseline_ts_file = "./value_strategy_performance_timeseries_exp_03_Filter 3.json"

with open(baseline_perf_file, "r") as f:
    baseline_perf = json.load(f)

with open(baseline_ts_file, "r") as f:
    baseline_ts = json.load(f)

# === Construct Performance Ratio DataFrame ===
metrics = sorted({m for run in runs for m in runs_data[run]["performance_ratio"].keys()})
perf_df = pd.DataFrame({run: runs_data[run]["performance_ratio"] for run in runs}).T[metrics]

# Add baseline to DataFrame
baseline_dict = {
    "Calmar Ratio": baseline_perf.get("Calmar Ratio"),
    "Sharpe Ratio": baseline_perf.get("Sharpe Ratio"),
    "Sortino Ratio": baseline_perf.get("Sortino Ratio"),
    "Max Drawdown": baseline_perf.get("Max Drawdown"),
    "Annualized Returns": baseline_perf.get("Annualized Returns"),
    "Annualized Volatility": baseline_perf.get("Annualized Volatility"),
    "Downside Volatility": baseline_perf.get("Downside Volatility"),
    "Average Stocks": baseline_perf.get("Average Stocks"),
    "Min Stocks": baseline_perf.get("Min Stocks"),
    "Max Stocks": baseline_perf.get("Max Stocks"),
}
for k in baseline_dict.keys():
    if k not in perf_df.columns:
        perf_df[k] = np.nan
perf_df.loc["Baseline"] = baseline_dict

preferred_order = [
    "Annualized Returns",
    "Annualized Volatility",
    "Downside Volatility",
    "Sharpe Ratio",
    "Sortino Ratio",
    "Calmar Ratio",
    "Max Drawdown",
    "Average Stocks",
    "Min Stocks",
    "Max Stocks"
]
perf_df = perf_df.reindex(columns=[c for c in preferred_order if c in perf_df.columns])

# === Construct Combined Time-Series DataFrame ===
max_len = max(len(runs_data[r]["plot_data"]["Final Investment"]) for r in runs)
ts_df = pd.DataFrame(index=range(max_len))
for run in runs:
    ts_df[run] = pd.Series(runs_data[run]["plot_data"]["Final Investment"])

# Add baseline time series (truncate or extend to same length)
baseline_series = pd.Series(baseline_ts["Final Investment"])
ts_df["Baseline"] = baseline_series.reindex(range(max_len)).ffill()

# === Plot 1: Combined Raw Portfolio Values ===
fig_raw = go.Figure()

# Plot experimental runs
for run in runs:
    fig_raw.add_trace(go.Scatter(
        y=ts_df[run],
        mode="lines",
        name=run,
        line=dict(width=1)
    ))

# Add baseline in a distinct color and thicker line
fig_raw.add_trace(go.Scatter(
    y=ts_df["Baseline"],
    mode="lines",
    name="Baseline",
    line=dict(color="red", width=3, dash="dash")
))

fig_raw.update_layout(
    title="💰 Portfolio Growth – All Runs vs Baseline",
    xaxis_title="Time Step",
    yaxis_title="Portfolio Value",
    hovermode="x unified",
    template="plotly_white"
)
fig_raw.show()
fig_raw.write_html("./output/plots/portfolio_growth_raw.html")

# === Plot 2: Normalized Portfolio Growth ===
norm_df = ts_df.div(ts_df.iloc[0])

fig_norm = go.Figure()
for run in runs:
    fig_norm.add_trace(go.Scatter(
        y=norm_df[run],
        mode="lines",
        name=run,
        line=dict(width=1)
    ))

# Add baseline normalized curve
fig_norm.add_trace(go.Scatter(
    y=norm_df["Baseline"],
    mode="lines",
    name="Baseline",
    line=dict(color="red", width=3, dash="dash")
))

fig_norm.update_layout(
    title="📈 Normalized Portfolio Growth – All Runs vs Baseline (Start = 1)",
    xaxis_title="Time Step",
    yaxis_title="Normalized Portfolio Value",
    hovermode="x unified",
    template="plotly_white"
)
fig_norm.show()
fig_norm.write_html("./output/plots/portfolio_growth_normalized.html")

# === Plot 3: Performance Metrics Bar Chart ===
metrics = perf_df.columns.tolist()
for metric in metrics:
    fig_metric = go.Figure()

    fig_metric.add_trace(go.Bar(
        x=[r for r in runs],
        y=[perf_df.loc[r, metric] for r in runs],
        name="Experiment Runs",
        marker_color="steelblue"
    ))
    fig_metric.add_trace(go.Bar(
        x=["Baseline"],
        y=[perf_df.loc["Baseline", metric]],
        name="Baseline",
        marker_color="crimson"
    ))
    fig_metric.update_layout(
        title=f"🏆 {metric} Comparison – Runs vs Baseline",
        xaxis_title="Run",
        yaxis_title=metric,
        barmode="group",
        template="plotly_white"
    )
    fig_metric.show()
    fig_metric.write_html(f"./output/plots/{metric.replace(' ', '_')}_comparison.html")


# === Plot 4: Heatmap – Relative Metric Strength ===
norm_perf = perf_df.copy().astype(float)
for col in norm_perf.columns:
    col_min = norm_perf[col].min()
    col_max = norm_perf[col].max()
    if col_max != col_min:
        norm_perf[col] = (norm_perf[col] - col_min) / (col_max - col_min)
    else:
        norm_perf[col] = 0.5

fig_heat = px.imshow(
    norm_perf.T,
    color_continuous_scale="Viridis",
    labels=dict(x="Run", y="Metric", color="Relative Performance"),
    title="🔥 Relative Performance Across Metrics (Baseline Included)"
)
fig_heat.update_layout(
    xaxis_tickangle=-45,
    template="plotly_white"
)
fig_heat.show()
fig_heat.write_html("./output/plots/performance_heatmap.html")

## Common Stocks

In [ ]:
# === METRIC STORAGE ===
rows = []
all_stocks_exp1, all_stocks_exp2 = set(), set()
filter_key = "Filter 3"
for month in common_months:
    f3_1 = set(exp1[month].get(filter_key, []))
    f3_2 = set(exp2[month].get(filter_key, []))

    all_stocks_exp1 |= f3_1
    all_stocks_exp2 |= f3_2

    common = f3_1 & f3_2
    union = f3_1 | f3_2
    unique = union - common

    jaccard = len(common) / len(union) if union else 0
    turnover = 1 - jaccard
    size_diff = abs(len(f3_1) - len(f3_2))

    rows.append({
        "Month": month,
        "Exp1_Count": len(f3_1),
        "Exp2_Count": len(f3_2),
        "Common_Stocks": len(common),
        "Unique_Stocks": len(unique),
        "Jaccard_Similarity": round(jaccard, 3),
        "Turnover_Rate": round(turnover, 3),
        "Portfolio_Size_Diff": size_diff
    })

# === CONVERT TO DATAFRAME ===
df = pd.DataFrame(rows)

# === AVERAGE METRICS ===
avg_common = df["Common_Stocks"].mean()
avg_unique = df["Unique_Stocks"].mean()
avg_jaccard = df["Jaccard_Similarity"].mean()
avg_turnover = df["Turnover_Rate"].mean()
avg_size_diff = df["Portfolio_Size_Diff"].mean()

# Persistent (always selected in both exps)
persistent_stocks = set.intersection(
    *[set(exp1[m].get(filter_key, [])) & set(exp2[m].get(filter_key, [])) for m in common_months]
) if common_months else set()

# Total unique stocks across both experiments
total_unique_all_time = len(all_stocks_exp1 | all_stocks_exp2)

# === PRINT SUMMARY ===
print(f"Total months compared: {len(common_months)}")
print(f"Average common stocks per month: {avg_common:.2f}")
print(f"Average unique stocks per month: {avg_unique:.2f}")
print(f"Average Jaccard similarity: {avg_jaccard:.3f}")
print(f"Average turnover rate: {avg_turnover:.3f}")
print(f"Average portfolio size difference: {avg_size_diff:.2f}")
print(f"Persistent stocks (selected in both exps every month): {list(persistent_stocks)}")
print(f"Total unique stocks across both exps: {total_unique_all_time}")

# === SAVE CSV ==
output_csv = "comparison_metrics.csv"
df.to_csv(output_csv, index=False)
print(f"\nDetailed month-wise comparison saved to: {output_csv}")